# Environment

Check that the OpenAI client works

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# To load OPEN AI_API_KEY from .env file
from openai import OpenAI
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [3]:
# To load GROQ_API_KEY from .env file 
from groq import Groq
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# groq_client = OpenAI(
#     api_key=os.getenv("GROQ_API_KEY"),
#     base_url="https://api.groq.com/openai/v1"
# )

* If you're writing a new Groq-only application → use:

    ```python
    from groq import Groq
    groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
    ```

* If you're building a provider-agnostic application or already use the OpenAI SDK → use:

    ```python
    from openai import OpenAI
    groq_client = OpenAI(
        api_key=os.getenv("GROQ_API_KEY"),
        base_url="https://api.groq.com/openai/v1"
    )
    ```

Both approaches are valid; they just serve different purposes.


# RAG

In [4]:
def openaillm(prompt):
    response = openai_client.responses.create(
       
        model= "gpt-5.4-mini",
        input = prompt,
    )
    return response.text

In [5]:
#openaillm("What is the capital of France?")

With OPEN_API KEY, I faced RateLimitError. Lets use Groq API

In [6]:
def groqllm(prompt):
    response = groq_client.chat.completions.create(
        model = "openai/gpt-oss-120b",
        messages = [
            {"role": "user", "content": prompt}
        ],
        
    )
    return response.choices[0].message.content


In [7]:
groqllm("What is the capital of France?")

'The capital of France is **Paris**.'

In [8]:
question = "I just discovered the course. Can I join now?"
answer = groqllm(question)
print(answer)

# It gives general response

Absolutely! Our courses are open‑enrollment, so you can sign up at any time—provided you meet any prerequisite requirements (if there are any for the specific class you’re interested in). Here’s a quick guide to get you started:

1. **Check the Course Details**  
   - Review the syllabus, required materials, and any recommended background knowledge.  
   - Make sure the class schedule (live sessions, assignment deadlines, etc.) fits your availability.

2. **Confirm Prerequisites** *(if applicable)*  
   - Some advanced courses may require prior completion of foundational modules or a certain level of experience. The course description will list those requirements.

3. **Enroll Through the Platform**  
   - Log in to your account (or create one if you haven’t already).  
   - Navigate to the **“Catalog”** or **“All Courses”** section, find the course, and click **“Enroll/Join.”**  
   - You’ll usually receive a confirmation email with next‑step instructions.

4. **Payment (if required)*

Build a prompt that includes both the question and the context:

In [9]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [10]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [11]:
answer = groqllm(prompt)
print(answer)

# It gives correct answer based on the context

Yes—you can still join the course. If you’d like to earn a certificate, just make sure you submit your project before the submission deadline. Otherwise you can start learning and doing the homework right away.


In [ ]:
documents = [
    "Groq provides ultra-fast AI inference.",
    "Python is a popular programming language.",
    "RAG stands for Retrieval-Augmented Generation.",
    "LangChain is a framework for LLM applications."
]

def retrieve(query):
    query_words = set(query.lower().split())

    scored_docs = []

    for doc in documents:
        print(f"Document: {doc}")
        doc_words = set(doc.lower().split())
        print(f"Document words: {doc_words}")
        score = len(query_words.intersection(doc_words))
        print(f"Score for document: {score}\n")
        scored_docs.append((score, doc))
        print(f"Scored documents so far: {scored_docs}\n")

    scored_docs.sort(reverse=True)

    return [doc for score, doc in scored_docs[:2] if score > 0]

def ask_rag(question):
    context = "\n".join(retrieve(question))
    print(f"Retrieved context:\n{context}\n")
    prompt = f"""
        Answer the question using only the provided context.

        Context:
        {context}

        Question:
        {question}
        """

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content

print(ask_rag("What is RAG ?"))

Document: Groq provides ultra-fast AI inference.
Document words: {'groq', 'provides', 'inference.', 'ai', 'ultra-fast'}
Score for document: 0

Scored documents so far: [(0, 'Groq provides ultra-fast AI inference.')]

Document: Python is a popular programming language.
Document words: {'python', 'a', 'language.', 'is', 'popular', 'programming'}
Score for document: 1

Scored documents so far: [(0, 'Groq provides ultra-fast AI inference.'), (1, 'Python is a popular programming language.')]

Document: RAG stands for Retrieval-Augmented Generation.
Document words: {'for', 'retrieval-augmented', 'stands', 'generation.', 'rag'}
Score for document: 1

Scored documents so far: [(0, 'Groq provides ultra-fast AI inference.'), (1, 'Python is a popular programming language.'), (1, 'RAG stands for Retrieval-Augmented Generation.')]

Document: LangChain is a framework for LLM applications.
Document words: {'llm', 'for', 'applications.', 'a', 'is', 'langchain', 'framework'}
Score for document: 1

Scor

```python
documents = [
    "Groq provides ultra-fast AI inference",
    "Python is a popular programming language",
    "RAG stands for Retrieval Augmented Generation"
]

query = "What is RAG"
```

### Step 1: Query words

```python
query_words = set(query.lower().split())
```

Result:

```python
{'what', 'is', 'rag'}
```

---

### Step 2: Score each document

For document:

```python
"Python is a popular programming language"
```

```python
doc_words = {
    'python', 'is', 'a', 'popular',
    'programming', 'language'
}
```

Intersection:

```python
query_words.intersection(doc_words)

# {'is'}
```

Score:

```python
len({'is'}) == 1
```

So:

```python
score = 1
```

---

For document:

```python
"RAG stands for Retrieval Augmented Generation"
```

```python
doc_words = {
    'rag', 'stands', 'for',
    'retrieval', 'augmented', 'generation'
}
```

Intersection:

```python
{'rag'}
```

Score:

```python
1
```

---

For document:

```python
"Groq provides ultra-fast AI inference"
```

Intersection:

```python
set()
```

Score:

```python
0
```

---

### What does score mean?

This line:

```python
score = len(query_words.intersection(doc_words))
```

means:

> Count how many words from the query also appear in the document.

Example:

| Query Words | Document Words           | Common Words | Score |
| ----------- | ------------------------ | ------------ | ----- |
| what is rag | python is language       | is           | 1     |
| what is rag | rag stands for retrieval | rag          | 1     |
| what is rag | groq ai inference        | none         | 0     |

---

### Why sort?

After scoring:

```python
scored_docs = [
    (0, "Groq provides ultra-fast AI inference"),
    (1, "Python is a popular programming language"),
    (1, "RAG stands for Retrieval Augmented Generation")
]
```

Sorting:

```python
scored_docs.sort(reverse=True)
```

gives:

```python
[
    (1, "RAG stands for Retrieval Augmented Generation"),
    (1, "Python is a popular programming language"),
    (0, "Groq provides ultra-fast AI inference")
]
```

Highest scores first.

---

### Why `score > 0`?

This line:

```python
return [doc for score, doc in scored_docs[:2] if score > 0]
```

means:

> Return only documents that matched at least one query word.

Without it:

```python
return [doc for score, doc in scored_docs[:2]]
```

you could get irrelevant documents.

Example:

Query:

```python
"What is RAG"
```

Scores:

```python
[
    (1, "RAG stands for Retrieval Augmented Generation"),
    (0, "Groq provides ultra-fast AI inference"),
    (0, "Python is a popular programming language")
]
```

Without `score > 0`, the second document would still be returned even though it has nothing to do with the query.

With `score > 0`:

```python
[
    "RAG stands for Retrieval Augmented Generation"
]
```

Only relevant matches are returned.

---

### Limitation of this approach

This is a **very naive keyword search**.

For example:

```python
query = "What is Retrieval Augmented Generation?"
```

The query words become:

```python
{'what', 'is', 'retrieval', 'augmented', 'generation'}
```

A document containing all three keywords gets score:

```python
3
```

while a document containing only `"retrieval"` gets:

```python
1
```

So higher scores generally indicate more relevance.

However, it won't understand meaning:

```python
query = "What is a car?"
document = "An automobile is a vehicle."
```

Score:

```python
0
```

because `"car"` and `"automobile"` are different words.

That's one reason embedding-based RAG is often used: it can match semantic similarity, not just exact word overlap.


# Dataset

In [13]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()
print(courses_raw)

[{'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 402}, {'course': 'stock-markets-analytics-zoomcamp', 'course_name': 'Stock Markets Analytics Zoomcamp', 'path': '/json/stock-markets-analytics-zoomcamp.json', 'questions_count': 93}, {'course': 'ai-dev-tools-zoomcamp', 'course_name': 'AI Dev Tools Zoomcamp', 'path': '/json/ai-dev-tools-zoomcamp.json', 'questions_count': 41}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 79}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 255}, {'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 472}]


In [14]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""
    print(f"Fetching course data from: {course_url}")
    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

Fetching course data from: https://datatalks.club/faq/json/data-engineering-zoomcamp.json
Fetching course data from: https://datatalks.club/faq/json/stock-markets-analytics-zoomcamp.json
Fetching course data from: https://datatalks.club/faq/json/ai-dev-tools-zoomcamp.json
Fetching course data from: https://datatalks.club/faq/json/llm-zoomcamp.json
Fetching course data from: https://datatalks.club/faq/json/mlops-zoomcamp.json
Fetching course data from: https://datatalks.club/faq/json/machine-learning-zoomcamp.json


1342

In [15]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

# Search

Search basics

At its core, every search engine does the same thing. It takes a query,
scores every document for similarity, and returns the top results.

Formally, there is a similarity function:

```python
score = sim(query, document)
```

For each document in the database, you compute this score. Then you
rank all documents by score and return the top N. What makes a search
engine different from another search engine is what `sim` actually
computes.

- text/lexical search (covered in this section): `sim` counts how
  many words the query and the document share. It looks at the surface
  form, the actual words, and matches them exactly.

- vector/semantic search (covered in [module 2](../../02-vector-search/)):
  `sim` compares the meaning of the query and the document. Same
  function, different similarity measure.

Consider these two questions:

- "Can I still join the course after the start date?"
- "Is it possible to enroll late?"

They mean the same thing, but share almost no keywords. "Join" is not
"enroll", "course" is absent, "start date" is not "late". A text search
engine would struggle to match them, because it only sees words.

We'll see how vector search solves this later. For now, let's build text
search with minsearch.

## Indexing with minsearch

We already have the `documents` list. Now
let's index it.

Searching matters because we have around 1100 documents. Sending all
of them to the LLM would be expensive and slow. The model would get
confused with too much data. Search finds the most relevant documents
to send instead.

There are many search libraries you can use - Apache Lucene,
Elasticsearch, Solr, and others. But these are somewhat heavy. For
example, to run Elasticsearch, you need to start a Docker container.

[minsearch](https://github.com/alexeygrigorev/minsearch) is a simple
in-memory search engine. It's lightweight, so it runs anywhere Python
runs, including Google Colab where you can't start a Docker container.
It's a toy implementation, not production ready, but it illustrates how
search engines work and it gives good results.

This library is maintained by Alexey Grigorev. [code](https://github.com/alexeygrigorev/build-your-own-search-engine).

The concepts in minsearch are the same as in Elasticsearch (which
comes from Lucene): text fields, keyword fields, boosting, filtering. 

We'll index the `question`, `section`, and `answer` fields as text
(they'll be tokenized and ranked), and the `course` field as a
keyword (for filtering).

The index tokenizes text fields and treats keyword fields as exact strings.

Text fields are the fields you search through. When you type a query,
the search engine looks at these fields and tokenizes them. It splits
text into words, lowercases them, removes stop words, and ranks the
results by relevance. So `question`, `section`, and `answer` are text
fields.

Keyword fields are for exact matching. Think of a SQL query like
`SELECT * FROM index WHERE course = 'data-engineering-zoomcamp'`. The
results must come from the specified course, no matter what ranking or
boosting you do for text fields.

You use keyword fields to restrict the search space to a particular
subset. In our case, we have four courses. Say you're taking the LLM
course and ask a question. You don't want answers from the MLOps or
machine learning courses mixed in.

```python
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)
```

That's it, the index is built. The `fit` name comes from scikit-learn,
where you fit a model on data. Here we fit an index on our documents.


In [16]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

## Boosting fields 
Not all fields are equally important. The `question` field is usually
more relevant than `section` for matching. Query words appearing in the
question is a stronger signal than them appearing in the section name.

## Filtering by course
Sometimes you want to restrict the search to a specific course.
`minsearch` supports keyword filtering:

In [17]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [18]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )